In [1]:
!pip install -U "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html


Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.5/150.5 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 59.8 MB/s eta 0:00:00
  Attempting uninstall: jax-cuda12-pjrt
    Found existing installation: jax-cuda12-pjrt 0.7.2
    Uninstalling jax-cuda12-pjrt-0.7.2:
      Successfully uninstalled jax-cuda12-pjrt-0.7.2
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82
  Attempting uninstall: jax-cuda12-plugin
    Found existing installation: jax-cuda12-plugin 0.7.2
    Uninstalling

In [2]:
import jax
import jax.numpy as jnp

print("JAX version:", jax.__version__)
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())

x = jnp.ones((1024, 1024))
y = jnp.dot(x, x)
print("dot shape:", y.shape)


JAX version: 0.8.1
Backend: gpu
Devices: [CudaDevice(id=0)]
dot shape: (1024, 1024)


In [3]:
import numpy as np
import jax
import jax.numpy as jnp
from functools import partial

jax.config.update("jax_enable_x64", True)

# ----------------------------
# 1. Gauss–Legendre on [-π, π]
# ----------------------------

def gauss_legendre_nodes_weights(K: int):
    x, w = np.polynomial.legendre.leggauss(K)  # host-side once
    phi = np.pi * x
    wphi = np.pi * w
    return jnp.asarray(phi), jnp.asarray(wphi)

# ----------------------------
# 2. Local U(1)+θ tensor
# ----------------------------

@jax.jit
def make_u1_theta_tensor(beta, theta, phi_nodes, w_phi):
    """
    T[r,u,l,d] ∝ exp[ β cos p + i (θ/2π) q ],
      p = φ_r + φ_u − φ_l − φ_d,
      q = (p mod 2π) ∈ [−π, π].
    """
    beta = jnp.asarray(beta)
    theta = jnp.asarray(theta)

    phi_r, phi_u, phi_l, phi_d = jnp.meshgrid(
        phi_nodes, phi_nodes, phi_nodes, phi_nodes, indexing="ij"
    )
    w_r, w_u, w_l, w_d = jnp.meshgrid(
        w_phi, w_phi, w_phi, w_phi, indexing="ij"
    )

    two_pi = 2.0 * jnp.pi
    p = phi_r + phi_u - phi_l - phi_d
    q = (p + jnp.pi) % two_pi - jnp.pi

    weight = jnp.exp(beta * jnp.cos(p) + 1j * theta * q / two_pi)
    pref = jnp.sqrt(w_r * w_u * w_l * w_d) / (two_pi ** 2)
    return pref * weight  # (K,K,K,K), complex128

# ----------------------------
# 3. Single TRG step (Levin–Nave)
# ----------------------------

@partial(jax.jit, static_argnums=(1,))
def trg_step(T, Dcut: int):
    """
    One TRG step for T[r,u,l,d]; bond dimension truncated to Dcut.
    """
    D = T.shape[0]
    assert T.shape == (D, D, D, D)
    assert Dcut <= D * D
    D_new = Dcut

    # Ma[(l,u),(r,d)] = T[r,u,l,d]
    T_lurd = jnp.transpose(T, (2, 1, 0, 3))
    Ma = jnp.reshape(T_lurd, (D * D, D * D))

    # Mb[(l,d),(r,u)] = T[r,u,l,d]
    T_ldru = jnp.transpose(T, (2, 3, 0, 1))
    Mb = jnp.reshape(T_ldru, (D * D, D * D))

    # --- SVD Ma -> S1, S3 ---
    Ua, sa, Vha = jnp.linalg.svd(Ma, full_matrices=False)
    Ua = Ua[:, :D_new]
    sa = sa[:D_new]
    Vha = Vha[:D_new, :]

    sqrt_sa = jnp.sqrt(sa)
    sqrt_sa_b = sqrt_sa[None, None, :]

    Ua_rs = jnp.reshape(Ua, (D, D, D_new))          # (w,a,r)
    Vha_rs = jnp.reshape(Vha, (D_new, D, D))        # (m,b,g)

    S1 = Ua_rs * sqrt_sa_b                          # S1[w,a,r]
    S3 = jnp.transpose(Vha_rs, (1, 2, 0)) * sqrt_sa_b  # S3[b,g,l]

    # --- SVD Mb -> S2, S4 ---
    Ub, sb, Vhb = jnp.linalg.svd(Mb, full_matrices=False)
    Ub = Ub[:, :D_new]
    sb = sb[:D_new]
    Vhb = Vhb[:D_new, :]

    sqrt_sb = jnp.sqrt(sb)
    sqrt_sb_b = sqrt_sb[None, None, :]

    Ub_rs = jnp.reshape(Ub, (D, D, D_new))
    Vhb_rs = jnp.reshape(Vhb, (D_new, D, D))

    S2 = Ub_rs * sqrt_sb_b                          # S2[a,b,u]
    S4 = jnp.transpose(Vhb_rs, (1, 2, 0)) * sqrt_sb_b  # S4[g,w,d]

    # --- Contract S1..S4 → T_new ---
    # T_new[r,u,l,d] = Σ_{w,a,b,g} S1[w,a,r] S2[a,b,u] S3[b,g,l] S4[g,w,d]
    T_new = jnp.einsum("war,abu,bgl,gwd->ruld", S1, S2, S3, S4)
    return T_new

# ----------------------------
# 4. Z(β, θ) via TRG loop
# ----------------------------

def Z_TRG_u1(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    T0 = make_u1_theta_tensor(beta, theta, phi_nodes, w_phi)

    def body_fun(_, T):
        return trg_step(T, Dcut)

    T_final = jax.lax.fori_loop(0, no_iter, body_fun, T0)
    Z = jnp.sum(T_final)
    L = 2 ** no_iter
    return Z, L

Z_TRG_u1_jit = jax.jit(Z_TRG_u1, static_argnums=(4, 5))

# ----------------------------
# 5. F(θ) and χ_top
# ----------------------------

def free_energy_density(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    Z, L = Z_TRG_u1_jit(beta, theta, phi_nodes, w_phi, Dcut, no_iter)
    V = L * L
    F = -jnp.log(jnp.abs(Z)) / V
    return jnp.real(F)

def chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut: int, no_iter: int):
    F0 = free_energy_density(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    Fp = free_energy_density(beta, +h,  phi_nodes, w_phi, Dcut, no_iter)
    Fm = free_energy_density(beta, -h,  phi_nodes, w_phi, Dcut, no_iter)
    return (Fp - 2.0 * F0 + Fm) / (h * h)

def free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut: int, no_iter: int):
    thetas = jnp.asarray(thetas)
    F_single = lambda th: free_energy_density(beta, th, phi_nodes, w_phi, Dcut, no_iter)
    return jax.vmap(F_single)(thetas)

# ----------------------------
# 6. Example: use the T4
# ----------------------------

if __name__ == "__main__":
    # You can crank these up; start moderate to not explode SVD flops.
    K      = 16      # Gauss–Legendre points (local bond dim)
    Dcut   = 16      # TRG truncation (≤ K^2)
    no_iter = 4      # L = 2^no_iter

    beta = 0.0
    h    = 0.05

    phi_nodes, w_phi = gauss_legendre_nodes_weights(K)

    # Warm-up / compile
    Z0, L = Z_TRG_u1_jit(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    print(f"beta={beta}, theta=0.0 → L={L}, |Z|={float(jnp.abs(Z0)):.8f}")

    # θ-grid scan (vmapped)
    thetas = jnp.linspace(0.0, 2.0 * jnp.pi, 17)
    F_vals = free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut, no_iter)
    for th, Fth in zip(np.array(thetas), np.array(F_vals)):
        print(f"theta={th:6.3f}, F(θ)≈{Fth:.8e}")

    # χ_top estimate at θ=0
    chi_est = chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut, no_iter)
    print(f"Estimated χ_top(β={beta}) ≈ {float(chi_est):.8e}")


beta=0.0, theta=0.0 → L=16, |Z|=1.00000000
theta= 0.000, F(θ)≈2.08166817e-17
theta= 0.393, F(θ)≈4.01457233e-04
theta= 0.785, F(θ)≈1.61379320e-03
theta= 1.178, F(θ)≈3.65170361e-03
theta= 1.571, F(θ)≈6.55211133e-03
theta= 1.963, F(θ)≈1.03662224e-02
theta= 2.356, F(θ)≈1.51666972e-02
theta= 2.749, F(θ)≈2.09905941e-02
theta= 3.142, F(θ)≈3.47766667e-02
theta= 3.534, F(θ)≈2.09885978e-02
theta= 3.927, F(θ)≈1.51665023e-02
theta= 4.320, F(θ)≈1.03634386e-02
theta= 4.712, F(θ)≈6.55477505e-03
theta= 5.105, F(θ)≈3.64668849e-03
theta= 5.498, F(θ)≈1.61111567e-03
theta= 5.890, F(θ)≈4.01717244e-04
theta= 6.283, F(θ)≈3.05311332e-16
Estimated χ_top(β=0.0) ≈ 5.23750957e-03


In [7]:
import numpy as np
import jax
import jax.numpy as jnp
from functools import partial

jax.config.update("jax_enable_x64", True)

# ----------------------------
# 1. Gauss–Legendre on [-π, π]
# ----------------------------

def gauss_legendre_nodes_weights(K: int):
    x, w = np.polynomial.legendre.leggauss(K)  # host-side once
    phi = np.pi * x
    wphi = np.pi * w
    return jnp.asarray(phi), jnp.asarray(wphi)

# ----------------------------
# 2. Local U(1)+θ tensor
# ----------------------------

@jax.jit
def make_u1_theta_tensor(beta, theta, phi_nodes, w_phi):
    """
    T[r,u,l,d] ∝ exp[ β cos p + i (θ/2π) q ],
      p = φ_r + φ_u − φ_l − φ_d,
      q = (p mod 2π) ∈ [−π, π].
    """
    beta = jnp.asarray(beta)
    theta = jnp.asarray(theta)

    phi_r, phi_u, phi_l, phi_d = jnp.meshgrid(
        phi_nodes, phi_nodes, phi_nodes, phi_nodes, indexing="ij"
    )
    w_r, w_u, w_l, w_d = jnp.meshgrid(
        w_phi, w_phi, w_phi, w_phi, indexing="ij"
    )

    two_pi = 2.0 * jnp.pi
    p = phi_r + phi_u - phi_l - phi_d
    q = (p + jnp.pi) % two_pi - jnp.pi

    weight = jnp.exp(beta * jnp.cos(p) + 1j * theta * q / two_pi)
    pref = jnp.sqrt(w_r * w_u * w_l * w_d) / (two_pi ** 2)
    return pref * weight  # (K,K,K,K), complex128

# ----------------------------
# 3. Single TRG step (Levin–Nave)
# ----------------------------

@partial(jax.jit, static_argnums=(1,))
def trg_step(T, Dcut: int):
    """
    One TRG step for T[r,u,l,d]; bond dimension truncated to Dcut.
    """
    D = T.shape[0]
    assert T.shape == (D, D, D, D)
    assert Dcut <= D * D
    D_new = Dcut

    # Ma[(l,u),(r,d)] = T[r,u,l,d]
    T_lurd = jnp.transpose(T, (2, 1, 0, 3))
    Ma = jnp.reshape(T_lurd, (D * D, D * D))

    # Mb[(l,d),(r,u)] = T[r,u,l,d]
    T_ldru = jnp.transpose(T, (2, 3, 0, 1))
    Mb = jnp.reshape(T_ldru, (D * D, D * D))

    # --- SVD Ma -> S1, S3 ---
    Ua, sa, Vha = jnp.linalg.svd(Ma, full_matrices=False)
    Ua = Ua[:, :D_new]
    sa = sa[:D_new]
    Vha = Vha[:D_new, :]

    sqrt_sa = jnp.sqrt(sa)
    sqrt_sa_b = sqrt_sa[None, None, :]

    Ua_rs = jnp.reshape(Ua, (D, D, D_new))          # (w,a,r)
    Vha_rs = jnp.reshape(Vha, (D_new, D, D))        # (m,b,g)

    S1 = Ua_rs * sqrt_sa_b                          # S1[w,a,r]
    S3 = jnp.transpose(Vha_rs, (1, 2, 0)) * sqrt_sa_b  # S3[b,g,l]

    # --- SVD Mb -> S2, S4 ---
    Ub, sb, Vhb = jnp.linalg.svd(Mb, full_matrices=False)
    Ub = Ub[:, :D_new]
    sb = sb[:D_new]
    Vhb = Vhb[:D_new, :]

    sqrt_sb = jnp.sqrt(sb)
    sqrt_sb_b = sqrt_sb[None, None, :]

    Ub_rs = jnp.reshape(Ub, (D, D, D_new))
    Vhb_rs = jnp.reshape(Vhb, (D_new, D, D))

    S2 = Ub_rs * sqrt_sb_b                          # S2[a,b,u]
    S4 = jnp.transpose(Vhb_rs, (1, 2, 0)) * sqrt_sb_b  # S4[g,w,d]

    # --- Contract S1..S4 → T_new ---
    # T_new[r,u,l,d] = Σ_{w,a,b,g} S1[w,a,r] S2[a,b,u] S3[b,g,l] S4[g,w,d]
    T_new = jnp.einsum("war,abu,bgl,gwd->ruld", S1, S2, S3, S4)
    return T_new

# ----------------------------
# 4. Z(β, θ) via TRG loop
# ----------------------------

def Z_TRG_u1(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    T0 = make_u1_theta_tensor(beta, theta, phi_nodes, w_phi)

    def body_fun(_, T):
        return trg_step(T, Dcut)

    T_final = jax.lax.fori_loop(0, no_iter, body_fun, T0)
    Z = jnp.sum(T_final)
    L = 2 ** no_iter
    return Z, L

Z_TRG_u1_jit = jax.jit(Z_TRG_u1, static_argnums=(4, 5))

# ----------------------------
# 5. F(θ) and χ_top
# ----------------------------

def free_energy_density(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    Z, L = Z_TRG_u1_jit(beta, theta, phi_nodes, w_phi, Dcut, no_iter)
    V = L * L
    F = -jnp.log(jnp.abs(Z)) / V
    return jnp.real(F)

def chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut: int, no_iter: int):
    F0 = free_energy_density(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    Fp = free_energy_density(beta, +h,  phi_nodes, w_phi, Dcut, no_iter)
    Fm = free_energy_density(beta, -h,  phi_nodes, w_phi, Dcut, no_iter)
    return (Fp - 2.0 * F0 + Fm) / (h * h)

def free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut: int, no_iter: int):
    thetas = jnp.asarray(thetas)
    F_single = lambda th: free_energy_density(beta, th, phi_nodes, w_phi, Dcut, no_iter)
    return jax.vmap(F_single)(thetas)

# ----------------------------
# 6. Example: use the T4
# ----------------------------

if __name__ == "__main__":
    # You can crank these up; start moderate to not explode SVD flops.
    K      = 16      # Gauss–Legendre points (local bond dim)
    Dcut   = 16      # TRG truncation (≤ K^2)
    no_iter = 4      # L = 2^no_iter

    beta = 0.0
    h    = 0.05

    phi_nodes, w_phi = gauss_legendre_nodes_weights(K)

    # Warm-up / compile
    Z0, L = Z_TRG_u1_jit(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    print(f"beta={beta}, theta=0.0 → L={L}, |Z|={float(jnp.abs(Z0)):.8f}")

    # θ-grid scan (vmapped)
    thetas = jnp.linspace(0.0, 2.0 * jnp.pi, 17)
    F_vals = free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut, no_iter)
    for th, Fth in zip(np.array(thetas), np.array(F_vals)):
        print(f"theta={th:6.3f}, F(θ)≈{Fth:.8e}")

    # χ_top estimate at θ=0
    chi_est = chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut, no_iter)
    print(f"Estimated χ_top(β={beta}) ≈ {float(chi_est):.8e}")

beta=0.0, theta=0.0 → L=16, |Z|=1.00000000
theta= 0.000, F(θ)≈2.08166817e-17
theta= 0.393, F(θ)≈4.01457233e-04
theta= 0.785, F(θ)≈1.61379320e-03
theta= 1.178, F(θ)≈3.65170361e-03
theta= 1.571, F(θ)≈6.55211133e-03
theta= 1.963, F(θ)≈1.03662224e-02
theta= 2.356, F(θ)≈1.51666972e-02
theta= 2.749, F(θ)≈2.09905941e-02
theta= 3.142, F(θ)≈3.47766667e-02
theta= 3.534, F(θ)≈2.09885978e-02
theta= 3.927, F(θ)≈1.51665023e-02
theta= 4.320, F(θ)≈1.03634386e-02
theta= 4.712, F(θ)≈6.55477505e-03
theta= 5.105, F(θ)≈3.64668849e-03
theta= 5.498, F(θ)≈1.61111567e-03
theta= 5.890, F(θ)≈4.01717244e-04
theta= 6.283, F(θ)≈3.05311332e-16
Estimated χ_top(β=0.0) ≈ 5.23750957e-03


In [8]:
import numpy as np
import jax
import jax.numpy as jnp
from functools import partial

# Use 64-bit floats/complex for stability
jax.config.update("jax_enable_x64", True)

# ----------------------------
# 1. Gauss–Legendre on [-π, π]
# ----------------------------

def gauss_legendre_nodes_weights(K: int):
    """K-point Gauss–Legendre nodes/weights on [-π, π]."""
    x, w = np.polynomial.legendre.leggauss(K)  # CPU once
    phi = np.pi * x
    wphi = np.pi * w
    return jnp.asarray(phi), jnp.asarray(wphi)

# ----------------------------
# 2. Local U(1)+θ tensor
# ----------------------------

@jax.jit
def make_u1_theta_tensor(beta, theta, phi_nodes, w_phi):
    """
    Rank-4 tensor T[r,u,l,d] for 2D U(1) gauge theory with θ-term.

      p = φ_r + φ_u − φ_l − φ_d
      q = (p mod 2π) in [−π, π]
      T ∝ exp[ β cos p + i (θ / 2π) q ] * sqrt(weights) / (2π)^2
    """
    beta = jnp.asarray(beta)
    theta = jnp.asarray(theta)

    phi_r, phi_u, phi_l, phi_d = jnp.meshgrid(
        phi_nodes, phi_nodes, phi_nodes, phi_nodes, indexing="ij"
    )
    w_r, w_u, w_l, w_d = jnp.meshgrid(
        w_phi, w_phi, w_phi, w_phi, indexing="ij"
    )

    two_pi = 2.0 * jnp.pi
    p = phi_r + phi_u - phi_l - phi_d
    q = (p + jnp.pi) % two_pi - jnp.pi

    weight = jnp.exp(beta * jnp.cos(p) + 1j * theta * q / two_pi)
    pref = jnp.sqrt(w_r * w_u * w_l * w_d) / (two_pi ** 2)
    return pref * weight  # shape (K,K,K,K), complex128

# ----------------------------
# 3. Single TRG step (Levin–Nave)
# ----------------------------

@partial(jax.jit, static_argnums=(1,))
def trg_step(T, Dcut: int):
    """
    One TRG step on T[r,u,l,d]; bond dimension truncated to Dcut.
    """
    D = T.shape[0]
    assert T.shape == (D, D, D, D)
    assert Dcut <= D * D
    D_new = Dcut

    # Ma[(l,u),(r,d)] = T[r,u,l,d]
    T_lurd = jnp.transpose(T, (2, 1, 0, 3))
    Ma = jnp.reshape(T_lurd, (D * D, D * D))

    # Mb[(l,d),(r,u)] = T[r,u,l,d]
    T_ldru = jnp.transpose(T, (2, 3, 0, 1))
    Mb = jnp.reshape(T_ldru, (D * D, D * D))

    # --- SVD Ma -> S1, S3 ---
    Ua, sa, Vha = jnp.linalg.svd(Ma, full_matrices=False)
    Ua = Ua[:, :D_new]
    sa = sa[:D_new]
    Vha = Vha[:D_new, :]

    sqrt_sa = jnp.sqrt(sa)
    sqrt_sa_b = sqrt_sa[None, None, :]

    Ua_rs = jnp.reshape(Ua, (D, D, D_new))        # (w,a,r)
    Vha_rs = jnp.reshape(Vha, (D_new, D, D))      # (m,b,g)

    S1 = Ua_rs * sqrt_sa_b                        # S1[w,a,r]
    S3 = jnp.transpose(Vha_rs, (1, 2, 0)) * sqrt_sa_b  # S3[b,g,l]

    # --- SVD Mb -> S2, S4 ---
    Ub, sb, Vhb = jnp.linalg.svd(Mb, full_matrices=False)
    Ub = Ub[:, :D_new]
    sb = sb[:D_new]
    Vhb = Vhb[:D_new, :]

    sqrt_sb = jnp.sqrt(sb)
    sqrt_sb_b = sqrt_sb[None, None, :]

    Ub_rs = jnp.reshape(Ub, (D, D, D_new))
    Vhb_rs = jnp.reshape(Vhb, (D_new, D, D))

    S2 = Ub_rs * sqrt_sb_b                        # S2[a,b,u]
    S4 = jnp.transpose(Vhb_rs, (1, 2, 0)) * sqrt_sb_b  # S4[g,w,d]

    # --- Contract S1..S4 → T_new ---
    # T_new[r,u,l,d] = Σ_{w,a,b,g} S1[w,a,r] S2[a,b,u] S3[b,g,l] S4[g,w,d]
    T_new = jnp.einsum("war,abu,bgl,gwd->ruld", S1, S2, S3, S4)
    return T_new

# ----------------------------
# 4. Z(β, θ) via TRG loop
# ----------------------------

def Z_TRG_u1(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    T0 = make_u1_theta_tensor(beta, theta, phi_nodes, w_phi)

    def body_fun(_, T):
        return trg_step(T, Dcut)

    T_final = jax.lax.fori_loop(0, no_iter, body_fun, T0)
    Z = jnp.sum(T_final)
    L = 2 ** no_iter
    return Z, L

# JIT with Dcut, no_iter static
Z_TRG_u1_jit = jax.jit(Z_TRG_u1, static_argnums=(4, 5))

# ----------------------------
# 5. F(θ) and χ_top via autodiff
# ----------------------------

def free_energy_density(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    Z, L = Z_TRG_u1_jit(beta, theta, phi_nodes, w_phi, Dcut, no_iter)
    V = L * L
    F = -jnp.log(jnp.abs(Z)) / V
    return jnp.real(F)

def chi_top_autodiff(beta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    """
    χ_top = F''(0) via automatic differentiation.
    """
    def F_theta(theta):
        return free_energy_density(beta, theta, phi_nodes, w_phi, Dcut, no_iter)
    Fpp_0 = jax.grad(jax.grad(F_theta))(0.0)
    return Fpp_0

def free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut: int, no_iter: int):
    """
    Vectorized F(θ) over an array of thetas.
    """
    thetas = jnp.asarray(thetas)

    def F_single(th):
        return free_energy_density(beta, th, phi_nodes, w_phi, Dcut, no_iter)

    return jax.vmap(F_single)(thetas)

# ----------------------------
# 6. Demo run (edit numbers here to crank the GPU)
# ----------------------------

if __name__ == "__main__":
    print("JAX backend:", jax.default_backend())
    print("Devices:", jax.devices())

    # ---- parameters you can change ----
    beta   = 0.0   # start at strong coupling
    K      = 16    # Gauss–Legendre points (local bond dim)
    Dcut   = 16    # TRG truncation (≤ K^2); try 24 or 32 later
    no_iter = 4    # L = 2^no_iter (4→L=16, 5→L=32)
    # -----------------------------------

    phi_nodes, w_phi = gauss_legendre_nodes_weights(K)

    # Warm-up + check Z(0)
    Z0, L = Z_TRG_u1_jit(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    Z0, L = jax.device_get(Z0), int(L)
    print(f"\nbeta={beta}, theta=0.0 -> L={L}, |Z|={abs(Z0):.8f}")

    # F(θ) on a grid
    thetas = jnp.linspace(0.0, 2.0 * jnp.pi, 17)
    F_vals = free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut, no_iter)
    F_vals = jax.device_get(F_vals)

    print("\nθ grid and F(θ):")
    for th, Fth in zip(np.array(thetas), np.array(F_vals)):
        print(f"theta={th:6.3f}, F(θ)≈{Fth:.8e}")

    # χ_top via autodiff
    chi_ad = chi_top_autodiff(beta, phi_nodes, w_phi, Dcut, no_iter)
    chi_ad = float(jax.device_get(chi_ad))
    print(f"\nAutodiff χ_top(β={beta}) ≈ {chi_ad:.8e}")


JAX backend: gpu
Devices: [CudaDevice(id=0)]

beta=0.0, theta=0.0 -> L=16, |Z|=1.00000000

θ grid and F(θ):
theta= 0.000, F(θ)≈2.08166817e-17
theta= 0.393, F(θ)≈4.01457233e-04
theta= 0.785, F(θ)≈1.61379320e-03
theta= 1.178, F(θ)≈3.65170361e-03
theta= 1.571, F(θ)≈6.55211133e-03
theta= 1.963, F(θ)≈1.03662224e-02
theta= 2.356, F(θ)≈1.51666972e-02
theta= 2.749, F(θ)≈2.09905941e-02
theta= 3.142, F(θ)≈3.47766667e-02
theta= 3.534, F(θ)≈2.09885978e-02
theta= 3.927, F(θ)≈1.51665023e-02
theta= 4.320, F(θ)≈1.03634386e-02
theta= 4.712, F(θ)≈6.55477505e-03
theta= 5.105, F(θ)≈3.64668849e-03
theta= 5.498, F(θ)≈1.61111567e-03
theta= 5.890, F(θ)≈4.01717244e-04
theta= 6.283, F(θ)≈3.05311332e-16

Autodiff χ_top(β=0.0) ≈ nan


In [12]:
import numpy as np
import jax
import jax.numpy as jnp
from functools import partial

# Use 64-bit floats/complex for stability
jax.config.update("jax_enable_x64", True)

# ----------------------------
# 1. Gauss–Legendre on [-π, π]
# ----------------------------

def gauss_legendre_nodes_weights(K: int):
    """K-point Gauss–Legendre nodes/weights on [-π, π]."""
    x, w = np.polynomial.legendre.leggauss(K)  # CPU once
    phi = np.pi * x
    wphi = np.pi * w
    return jnp.asarray(phi), jnp.asarray(wphi)

# ----------------------------
# 2. Local U(1)+θ tensor
# ----------------------------

@jax.jit
def make_u1_theta_tensor(beta, theta, phi_nodes, w_phi):
    """
    Rank-4 tensor T[r,u,l,d] for 2D U(1) gauge theory with θ-term.

      p = φ_r + φ_u − φ_l − φ_d
      q = (p mod 2π) in [−π, π]
      T ∝ exp[ β cos p + i (θ / 2π) q ] * sqrt(weights) / (2π)^2
    """
    beta = jnp.asarray(beta)
    theta = jnp.asarray(theta)

    phi_r, phi_u, phi_l, phi_d = jnp.meshgrid(
        phi_nodes, phi_nodes, phi_nodes, phi_nodes, indexing="ij"
    )
    w_r, w_u, w_l, w_d = jnp.meshgrid(
        w_phi, w_phi, w_phi, w_phi, indexing="ij"
    )

    two_pi = 2.0 * jnp.pi
    p = phi_r + phi_u - phi_l - phi_d
    q = (p + jnp.pi) % two_pi - jnp.pi

    weight = jnp.exp(beta * jnp.cos(p) + 1j * theta * q / two_pi)
    pref = jnp.sqrt(w_r * w_u * w_l * w_d) / (two_pi ** 2)
    return pref * weight  # shape (K,K,K,K), complex128

# ----------------------------
# 3. Single TRG step (Levin–Nave)
# ----------------------------

@partial(jax.jit, static_argnums=(1,))
def trg_step(T, Dcut: int):
    """
    One TRG step on T[r,u,l,d]; bond dimension truncated to Dcut.
    """
    D = T.shape[0]
    assert T.shape == (D, D, D, D)
    assert Dcut <= D * D
    D_new = Dcut

    # Ma[(l,u),(r,d)] = T[r,u,l,d]
    T_lurd = jnp.transpose(T, (2, 1, 0, 3))
    Ma = jnp.reshape(T_lurd, (D * D, D * D))

    # Mb[(l,d),(r,u)] = T[r,u,l,d]
    T_ldru = jnp.transpose(T, (2, 3, 0, 1))
    Mb = jnp.reshape(T_ldru, (D * D, D * D))

    # --- SVD Ma -> S1, S3 ---
    Ua, sa, Vha = jnp.linalg.svd(Ma, full_matrices=False)
    Ua = Ua[:, :D_new]
    sa = sa[:D_new]
    Vha = Vha[:D_new, :]

    sqrt_sa = jnp.sqrt(sa)
    sqrt_sa_b = sqrt_sa[None, None, :]

    Ua_rs = jnp.reshape(Ua, (D, D, D_new))        # (w,a,r)
    Vha_rs = jnp.reshape(Vha, (D_new, D, D))      # (m,b,g)

    S1 = Ua_rs * sqrt_sa_b                        # S1[w,a,r]
    S3 = jnp.transpose(Vha_rs, (1, 2, 0)) * sqrt_sa_b  # S3[b,g,l]

    # --- SVD Mb -> S2, S4 ---
    Ub, sb, Vhb = jnp.linalg.svd(Mb, full_matrices=False)
    Ub = Ub[:, :D_new]
    sb = sb[:D_new]
    Vhb = Vhb[:D_new, :]

    sqrt_sb = jnp.sqrt(sb)
    sqrt_sb_b = sqrt_sb[None, None, :]

    Ub_rs = jnp.reshape(Ub, (D, D, D_new))
    Vhb_rs = jnp.reshape(Vhb, (D_new, D, D))

    S2 = Ub_rs * sqrt_sb_b                        # S2[a,b,u]
    S4 = jnp.transpose(Vhb_rs, (1, 2, 0)) * sqrt_sb_b  # S4[g,w,d]

    # --- Contract S1..S4 → T_new ---
    # T_new[r,u,l,d] = Σ_{w,a,b,g} S1[w,a,r] S2[a,b,u] S3[b,g,l] S4[g,w,d]
    T_new = jnp.einsum("war,abu,bgl,gwd->ruld", S1, S2, S3, S4)
    return T_new

# ----------------------------
# 4. Z(β, θ) via TRG loop
# ----------------------------

def Z_TRG_u1(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    T0 = make_u1_theta_tensor(beta, theta, phi_nodes, w_phi)

    def body_fun(_, T):
        return trg_step(T, Dcut)

    T_final = jax.lax.fori_loop(0, no_iter, body_fun, T0)
    Z = jnp.sum(T_final)
    L = 2 ** no_iter
    return Z, L

# JIT with Dcut, no_iter static
Z_TRG_u1_jit = jax.jit(Z_TRG_u1, static_argnums=(4, 5))

# ----------------------------
# 5. F(θ) and χ_top via finite difference
# ----------------------------

def free_energy_density(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    Z, L = Z_TRG_u1_jit(beta, theta, phi_nodes, w_phi, Dcut, no_iter)
    V = L * L
    F = -jnp.log(jnp.abs(Z)) / V
    return jnp.real(F)

def chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut: int, no_iter: int):
    """
    χ_top ≈ [F(h) - 2 F(0) + F(-h)] / h^2
    """
    F0 = free_energy_density(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    Fp = free_energy_density(beta, +h,  phi_nodes, w_phi, Dcut, no_iter)
    Fm = free_energy_density(beta, -h,  phi_nodes, w_phi, Dcut, no_iter)
    return (Fp - 2.0 * F0 + Fm) / (h * h)

def free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut: int, no_iter: int):
    thetas = jnp.asarray(thetas)

    def F_single(th):
        return free_energy_density(beta, th, phi_nodes, w_phi, Dcut, no_iter)

    return jax.vmap(F_single)(thetas)

# ----------------------------
# 6. Demo run (edit numbers here)
# ----------------------------

if __name__ == "__main__":
    print("JAX backend:", jax.default_backend())
    print("Devices:", jax.devices())

    # ---- parameters you can change ----
    beta   = 0.0   # strong coupling test
    K      = 16    # Gauss–Legendre points (local bond dim)
    Dcut   = 16    # TRG truncation (≤ K^2)
    no_iter = 4    # L = 2^no_iter (4→L=16, 5→L=32)
    h      = 0.05  # step size for χ_top finite difference
    # -----------------------------------

    phi_nodes, w_phi = gauss_legendre_nodes_weights(K)

    # Warm-up + check Z(0)
    Z0, L = Z_TRG_u1_jit(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    Z0, L = jax.device_get(Z0), int(L)
    print(f"\nbeta={beta}, theta=0.0 -> L={L}, |Z|={abs(Z0):.8f}")

    # F(θ) on a grid
    thetas = jnp.linspace(0.0, 2.0 * jnp.pi, 17)
    F_vals = free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut, no_iter)
    F_vals = jax.device_get(F_vals)

    print("\nθ grid and F(θ):")
    for th, Fth in zip(np.array(thetas), np.array(F_vals)):
        print(f"theta={th:6.3f}, F(θ)≈{Fth:.8e}")

    # χ_top via finite difference
    chi_fd = chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut, no_iter)
    chi_fd = float(jax.device_get(chi_fd))
    print(f"\nFinite-diff χ_top(β={beta}) ≈ {chi_fd:.8e}")

beta   = 0.0
K      = 24
Dcut   = 24
no_iter = 5
h      = 0.05


JAX backend: gpu
Devices: [CudaDevice(id=0)]

beta=0.0, theta=0.0 -> L=16, |Z|=1.00000000

θ grid and F(θ):
theta= 0.000, F(θ)≈2.08166817e-17
theta= 0.393, F(θ)≈4.01457233e-04
theta= 0.785, F(θ)≈1.61379320e-03
theta= 1.178, F(θ)≈3.65170361e-03
theta= 1.571, F(θ)≈6.55211133e-03
theta= 1.963, F(θ)≈1.03662224e-02
theta= 2.356, F(θ)≈1.51666972e-02
theta= 2.749, F(θ)≈2.09905941e-02
theta= 3.142, F(θ)≈3.47766667e-02
theta= 3.534, F(θ)≈2.09885978e-02
theta= 3.927, F(θ)≈1.51665023e-02
theta= 4.320, F(θ)≈1.03634386e-02
theta= 4.712, F(θ)≈6.55477505e-03
theta= 5.105, F(θ)≈3.64668849e-03
theta= 5.498, F(θ)≈1.61111567e-03
theta= 5.890, F(θ)≈4.01717244e-04
theta= 6.283, F(θ)≈3.05311332e-16

Finite-diff χ_top(β=0.0) ≈ 5.23750957e-03


In [13]:
# Parameter scan with Dcut = K so JAX fori_loop shapes stay consistent

beta = 0.0
h    = 0.05

def chi_top_est(beta, h, K, no_iter):
    # local bond dimension = K, truncation Dcut = K
    phi_nodes, w_phi = gauss_legendre_nodes_weights(K)
    Dcut = K
    # warm-up / compile for this configuration
    _ = free_energy_density(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    chi = chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut, no_iter)
    return float(jax.device_get(chi))

print(f"beta = {beta}, h = {h}")
for no_iter in [2, 3, 4, 5]:          # L = 4, 8, 16, 32
    for K in [8, 12, 16, 24]:
        L   = 2 ** no_iter
        chi = chi_top_est(beta, h, K, no_iter)
        print(f"L={L:3d}, K={K:2d}, Dcut={K:2d} -> chi_top ≈ {chi: .6e}")
beta   = 0.0
K      = 24
Dcut   = 24
no_iter = 5
h      = 0.05


beta = 0.0, h = 0.05
L=  4, K= 8, Dcut= 8 -> chi_top ≈  3.186381e-04
L=  4, K=12, Dcut=12 -> chi_top ≈  1.423481e-02
L=  4, K=16, Dcut=16 -> chi_top ≈  2.460992e-02
L=  4, K=24, Dcut=24 -> chi_top ≈  2.190078e-02
L=  8, K= 8, Dcut= 8 -> chi_top ≈  1.242179e-02
L=  8, K=12, Dcut=12 -> chi_top ≈  1.101285e-02
L=  8, K=16, Dcut=16 -> chi_top ≈  1.079993e-02
L=  8, K=24, Dcut=24 -> chi_top ≈  1.007139e-02
L= 16, K= 8, Dcut= 8 -> chi_top ≈  5.367037e-03
L= 16, K=12, Dcut=12 -> chi_top ≈  5.176672e-03
L= 16, K=16, Dcut=16 -> chi_top ≈  5.237510e-03
L= 16, K=24, Dcut=24 -> chi_top ≈  5.204265e-03
L= 32, K= 8, Dcut= 8 -> chi_top ≈  2.648494e-03
L= 32, K=12, Dcut=12 -> chi_top ≈  2.594957e-03
L= 32, K=16, Dcut=16 -> chi_top ≈  2.599144e-03
L= 32, K=24, Dcut=24 -> chi_top ≈  2.599039e-03
